# OCR Output Review Tool
Interactive review of Gemini OCR results with manual correction capabilities.
- Identify low-confidence text blocks (< 0.9)
- View original images for verification
- Correct OCR errors
- Save corrected results

In [10]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Image as IPImage, display, HTML
import os

print("Libraries imported successfully")

Libraries imported successfully


In [11]:
# Setup project paths
script_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
project_root = script_dir if (script_dir / "data").exists() else script_dir.parent

ocr_output_file = project_root / "data" / "02_raw_batch" / "ocr_output.jsonl"
preprocessed_dir = project_root / "data" / "01_preprocessed"
preprocessed_meta_csv = preprocessed_dir / "all_metadata.csv"
review_output_file = project_root / "data" / "02_raw_batch" / "ocr_output_reviewed.jsonl"

print(f"Project Root: {project_root}")
print(f"OCR Output File: {ocr_output_file}")
print(f"Preprocessed Data Dir: {preprocessed_dir}")
print(f"Review Output File: {review_output_file}")
print(f"\nFile exists: {ocr_output_file.exists()}")

Project Root: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities
OCR Output File: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\02_raw_batch\ocr_output.jsonl
Preprocessed Data Dir: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\01_preprocessed
Review Output File: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\02_raw_batch\ocr_output_reviewed.jsonl

File exists: True


In [12]:
meta_df = pd.read_csv(preprocessed_meta_csv)
print(f"Total metadata records loaded: {len(meta_df)}")
# meta_df['x_offset'] = pd.to_numeric(meta_df['x_offset'], errors='coerce')
# meta_df['y_offset'] = pd.to_numeric(meta_df['y_offset'], errors='coerce')
# print(type(meta_df['x_offset'][0]))
print(meta_df.head())

Total metadata records loaded: 3789
    pub_id        date  page_num  column  \
0  Alabama  0000-00-00         1       0   
1  Alabama  0000-00-00         1       1   
2  Alabama  0000-00-00         1       2   
3  Alabama  0000-00-00         2       0   
4  Alabama  0000-00-00         2       1   

                                                path  x_offset  y_offset  
0  data/01_preprocessed/Alabama/Alabama_p01_r00_c...         0      1533  
1  data/01_preprocessed/Alabama/Alabama_p01_r00_c...       867      1533  
2  data/01_preprocessed/Alabama/Alabama_p01_r00_c...      1547      1533  
3  data/01_preprocessed/Alabama/Alabama_p02_r00_c...         0       221  
4  data/01_preprocessed/Alabama/Alabama_p02_r00_c...       812       221  


In [13]:
# Load OCR output from JSONL
ocr_data = []
with open(ocr_output_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            ocr_data.append(json.loads(line))

df_ocr = pd.DataFrame(ocr_data)

print(f"Total OCR records loaded: {len(df_ocr)}")
print(f"\nDataframe shape: {df_ocr.shape}")
print(f"\nColumn names: {df_ocr.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_ocr.head())

Total OCR records loaded: 1271

Dataframe shape: (1271, 6)

Column names: ['pub', 'page', 'col', 'text', 'conf', 'bbox']

First few rows:
       pub  page  col                                text  conf  \
0  Alabama     1    0                             ALABAMA  0.95   
1  Alabama     1    0               ABANDA, 125, CHAMBERS  0.95   
2  Alabama     1    0    CLARK, JAMES THOMAS (b'76)--Ala.  0.95   
3  Alabama     1    0                       4,'11; (l'11)  0.95   
4  Alabama     1    0  Turk, Wm. Pellham-Ga.5,'92; (l'01)  0.95   

                                                bbox  
0  {'x': 556.0, 'y': 1554.0, 'width': 166.0, 'hei...  
1  {'x': 297.0, 'y': 1601.0, 'width': 434.0, 'hei...  
2  {'x': 297.0, 'y': 1622.0, 'width': 686.0, 'hei...  
3  {'x': 335.0, 'y': 1642.0, 'width': 168.0, 'hei...  
4  {'x': 296.0, 'y': 1662.0, 'width': 548.0, 'hei...  


In [14]:
# Identify low-confidence items
df_ocr['conf_numeric'] = pd.to_numeric(df_ocr['conf'], errors='coerce')

low_conf_threshold = 0.90
df_low_conf = df_ocr[df_ocr['conf_numeric'] < low_conf_threshold]

print(f"Confidence Score Statistics:")
print(df_ocr['conf_numeric'].describe())
print(f"\n---")
print(f"Records with confidence < {low_conf_threshold}: {len(df_low_conf)} out of {len(df_ocr)}")
print(f"Percentage: {100 * len(df_low_conf) / len(df_ocr):.4f}%")
print(f"\nConfidence distribution:")
print(df_ocr['conf_numeric'].value_counts().sort_index(ascending=True))


Confidence Score Statistics:
count    1271.000000
mean        0.970897
std         0.019988
min         0.950000
25%         0.950000
50%         0.990000
75%         0.990000
max         0.990000
Name: conf_numeric, dtype: float64

---
Records with confidence < 0.9: 0 out of 1271
Percentage: 0.0000%

Confidence distribution:
conf_numeric
0.95    607
0.99    664
Name: count, dtype: int64


In [15]:
df_low_conf.head()

,pub,page,col,text,conf,bbox,conf_numeric


In [16]:
# Helper function to find original image file
def find_original_image(state, page_num, col_idx):
    """
    Reconstruct the original image path from preprocessed directory.
    Images are stored as: data/01_preprocessed/{state}/{state}_p{page}_r00_c{col}.jpg
    """
    
    # Construct expected filename pattern
    img_path = preprocessed_dir / state /  f"{state}_p{page_num:02d}_r00_c{col_idx:03d}.jpg"
    if img_path.exists():
        return img_path
    
    return None

# Test the function
test_result = find_original_image(df_low_conf.iloc[0]['pub'] if len(df_low_conf) > 0 else 'Texas', 1, 0)
print(f"Test image search result: {test_result}")
print(f"Test path exists: {test_result.exists() if test_result else False}")

Test image search result: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\01_preprocessed\Texas\Texas_p01_r00_c000.jpg
Test path exists: True


In [17]:
def display_review_item(index):
    """Display a single low-confidence item with image and text fields for correction"""
    if index >= len(df_low_conf):
        print("✓ All items reviewed!")
        return
    
    row = df_low_conf.iloc[index]
    orig_index = df_low_conf.index[index]
    
    # Find original image
    img_path = find_original_image(row['pub'], row['page'], row['col'])
    
    print(f"\n{'='*80}")
    print(f"Item {index + 1} of {len(df_low_conf)}")
    print(f"{'='*80}")
    print(f"Publication: {row['pub']}, Page: {row['page']}, Column: {row['col']}")
    print(f"Confidence: {row['conf_numeric']:.4f}")
    print(f"Original Index: {orig_index}")
    print(f"OCR Text: {row['text']}")
       
    print(f"\nContext:")
    print(df_ocr.loc[orig_index - 3: orig_index + 4])
    # Display bounding box info
    bbox = dict(row['bbox'])
    print(f"Bounding Box: x={bbox['x']}, y={bbox['y']}, w={bbox['width']}, h={bbox['height']}")
    meta_row = meta_df.loc[
        (meta_df["pub_id"] == row['pub']) & 
        (meta_df["page_num"] == row['page']) & 
        (meta_df["column"] == row['col'])
    ][["x_offset", "y_offset"]]
    bbox['x'] -= float(meta_row['x_offset'].iloc[0])
    bbox['y'] -= float(meta_row['y_offset'].iloc[0])
    

    print(f"Bounding Box: x={bbox['x']}, y={bbox['y']}, w={bbox['width']}, h={bbox['height']}")
    
    # Display image if found
    if img_path:
        print(f'\n📷 Original Image: "{img_path}"')
        img = Image.open(img_path)
        plt.figure(figsize=(10, 12))
        fig, ax = plt.subplots()
        ax.imshow(img)
        plt.axis('off')
        rect = plt.Rectangle(
            (bbox['x'], bbox['y']),
            bbox['width'],
            bbox['height'],
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )

        ax.add_patch(rect)
        #ax.set_title(f"Original Column Image - {row['pub']} p{row['page']} c{row['col']}")
        ax.set_ylim(bbox['y'] + 100, bbox['y'] - bbox["height"] - 150)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️  Original image not found at expected location")
 

# Display first low-confidence item
display_review_item(4)

✓ All items reviewed!


In [19]:
# functions for interactions
def update_display(change=None):
    """Update display when item index changes"""
    idx = current_idx.value
    row = df_low_conf.iloc[idx]
    
    corrected_text.value = row['text']
    confidence_display.value = f"<b>Confidence: {row['conf_numeric']:.4f}</b> | Pub: {row['pub']} | Page: {row['page']} | Col: {row['col']}"
    status_display.value = f"<i>Item {idx + 1} of {len(df_low_conf)} | Reviewed: {review_state['reviewed_count']}</i>"

    context_display.value = f"""
    <b>Context:</b><br>
    {df_ocr.loc[df_low_conf.index[idx] - 4: df_low_conf.index[idx] - 1].to_html()}
    <b>{df_ocr.loc[[df_low_conf.index[idx]]].to_html()}</b>
    {df_ocr.loc[df_low_conf.index[idx] + 1: df_low_conf.index[idx] + 4].to_html()}
    """
    
    # Display image
    img_path = find_original_image(row['pub'], row['page'], row['col'])
    if img_path:
        with open(img_path, "rb") as file:
            image_bytes = file.read()
        image_widget.value = image_bytes
def on_save_correction(b):
    idx = current_idx.value
    orig_idx = df_low_conf.index[idx]
    review_state['corrections'][orig_idx] = corrected_text.value.strip()
    review_state['reviewed_count'] += 1
    if idx < len(df_low_conf) - 1:
        current_idx.value = idx + 1
    
def on_skip(b):
    idx = current_idx.value
    review_state['skipped'].add(idx)
    if idx < len(df_low_conf) - 1:
        current_idx.value = idx + 1

def on_previous(b):
    if current_idx.value > 0:
        current_idx.value = current_idx.value - 1

In [20]:
if len(df_low_conf) <= 0:
    print("No low confidence OCR recorded!")
else:
    # Create review state tracking dictionary
    review_state = {
        'current_index': 0,
        'corrections': {},  # Store {original_index: corrected_text}
        'skipped': set(),
        'reviewed_count': 0
    }

    # Create interactive review interface
    current_idx = widgets.IntSlider(value=0, min=0, max=len(df_low_conf)-1, step=1, description='Item:', width='500px')
    corrected_text = widgets.Text(value='', placeholder='Enter corrected text', description='Correction:', width='500px')
    confidence_display = widgets.HTML(value='')
    status_display = widgets.HTML(value='')
    context_display = widgets.HTML(value='')
    row = df_low_conf.iloc[0]
    img_path = find_original_image(row['pub'], row['page'], row['col'])
    if img_path:
        with open(img_path, "rb") as file:
            image_bytes = file.read()
        image_widget = widgets.Image(
            value=image_bytes,
            format='jpg',
            width=300,
            height=900,
            layout=widgets.Layout(object_fit='scale-down') # Optional styling
        )

    current_idx.observe(update_display, names='value')

    # Buttons for navigation and saving
    btn_save_correction = widgets.Button(description='Save & Next', button_style='success')
    btn_skip = widgets.Button(description='Skip', button_style='info')
    btn_previous = widgets.Button(description='Previous', button_style='warning')

    btn_save_correction.on_click(on_save_correction)
    btn_skip.on_click(on_skip)
    btn_previous.on_click(on_previous)
    buttons_box = widgets.HBox([btn_previous, btn_skip, btn_save_correction])

    control_box = widgets.VBox([
        confidence_display,
        status_display,
        current_idx,
        corrected_text,
        buttons_box,
        context_display
    ])

    # Display the interface
    update_display()
    display(widgets.HBox([control_box, image_widget]))

No low confidence OCR recorded!


In [54]:
# Summary of corrections
print("="*80)
print("REVIEW SUMMARY")
print("="*80)
print(f"Total low-confidence items: {len(df_low_conf)}")
print(f"Items reviewed & corrected: {len(review_state['corrections'])}")
print(f"Items skipped: {len(review_state['skipped'])}")
print(f"Items not yet reviewed: {len(df_low_conf) - len(review_state['corrections']) - len(review_state['skipped'])}")
# print("\nSample corrections made:")
# for orig_idx, corrected_text in list(review_state['corrections'].items())[:5]:
#     print(f"  Index {orig_idx}: {corrected_text[:60]}")

REVIEW SUMMARY
Total low-confidence items: 83
Items reviewed & corrected: 83
Items skipped: 1
Items not yet reviewed: -1


In [55]:
# Apply corrections to original data and save to new file
df_reviewed = df_ocr.copy()

# Apply corrections
for orig_idx, corrected_text in review_state['corrections'].items():
    df_reviewed.loc[orig_idx, 'text'] = corrected_text
    df_reviewed.loc[orig_idx, 'conf_numeric'] = 1.0  # Set confidence to 1.0 for corrected items

# Mark items as reviewed
#df_reviewed['reviewed'] = df_reviewed.index.isin(review_state['corrections'].keys())

print(f"Total records to save: {len(df_reviewed)}")
print(f"Records with corrections: {df_reviewed['conf_numeric'].eq(1.0).sum()}")
print(f"\nSample of reviewed data:")
print(df_reviewed[df_reviewed['conf_numeric'] == 1.0].head())

Total records to save: 345276
Records with corrections: 505

Sample of reviewed data:
              pub  page  col text  conf  \
2477      Alabama    10    0       0.85   
3760      Alabama    14    2       0.70   
7258     Arkansas     3    2   ;▼  0.85   
24748  California    45    2       0.50   
27237    Colorado     2    0    Y  0.85   

                                                    bbox  conf_numeric  
2477   {'x': 673.0, 'y': 1455.0, 'width': 4.0, 'heigh...           1.0  
3760   {'x': 1529.0, 'y': 1491.0, 'width': 160.0, 'he...           1.0  
7258   {'x': 1480.0, 'y': 541.0, 'width': 44.0, 'heig...           1.0  
24748  {'x': 1643.0, 'y': 1758.0, 'width': 5.0, 'heig...           1.0  
27237  {'x': 99.0, 'y': 1022.0, 'width': 15.0, 'heigh...           1.0  


In [56]:
# Save reviewed results to new JSONL file
def save_reviewed_output(df, output_path):
    """Save dataframe to JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for idx, row in df.iterrows():
            # Reconstruct the original entry format
            entry = {
                'pub': row['pub'],
                'page': row['page'],
                'col': row['col'],
                'text': row['text'],
                'conf': row['conf_numeric'] if isinstance(row['conf_numeric'], (int, float)) else float(row['conf_numeric']),
                'bbox': row['bbox']
            }
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    return output_path

# Save the reviewed data
output_path = save_reviewed_output(df_reviewed, review_output_file)
print(f"✓ Reviewed data saved to: {output_path}")
print(f"✓ Total records: {len(df_reviewed)}")
print(f"✓ Corrected records: {df_reviewed['conf_numeric'].eq(1.0).sum()}")

# Verify the output file
with open(output_path, 'r', encoding='utf-8') as f:
    sample_lines = [json.loads(line) for line in list(f)[:2]]
    
print(f"\n✓ Verification - First 2 records from new file:")
for i, entry in enumerate(sample_lines):
    print(f"\nRecord {i+1}:")
    print(f"  Text: {entry['text'][:60]}...")
    print(f"  Confidence: {entry['conf']}")
    print(f"  Reviewed: {entry.get('reviewed', False)}")

✓ Reviewed data saved to: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\02_raw\ocr_output_reviewed.jsonl
✓ Total records: 345276
✓ Corrected records: 505

✓ Verification - First 2 records from new file:

Record 1:
  Text: ALABAMA...
  Confidence: 0.99
  Reviewed: False

Record 2:
  Text: ABANDA, 125, CHAMBERS...
  Confidence: 0.99
  Reviewed: False


In [58]:
# Final statistics and comparison
print("\n" + "="*80)
print("FINAL STATISTICS")
print("="*80)

# Load both files for comparison
df_original = pd.read_json(ocr_output_file, lines=True)
df_final = pd.read_json(review_output_file, lines=True)

print(f"\nOriginal OCR Output:")
print(f"  Total records: {len(df_original)}")
print(f"  Low confidence (<0.90): {(df_original['conf'] < 0.90).sum()}")
print(f"  Average confidence: {df_original['conf'].mean():.4f}")

print(f"\nReviewed Output:")
print(f"  Total records: {len(df_final)}")
#print(f"  Reviewed records: {df_final['reviewed'].sum() if 'reviewed' in df_final.columns else 'N/A'}")
print(f"  Low confidence (<0.90): {(df_final['conf'] < 0.90).sum()}")
print(f"  Average confidence: {df_final['conf'].mean():.4f}")

print(f"\nOutput saved to: {review_output_file}")
print(f"Ready for next processing step!")


FINAL STATISTICS

Original OCR Output:
  Total records: 345276
  Low confidence (<0.90): 83
  Average confidence: 0.9836

Reviewed Output:
  Total records: 345276
  Low confidence (<0.90): 0
  Average confidence: 0.9836

Output saved to: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\02_raw\ocr_output_reviewed.jsonl
Ready for next processing step!
